Este notebook documenta a etapa de origem do projeto. A listagem mantida no TV Globo Wiki é lida como HTML e transformada em uma tabela com data, título e observações.

::: {.callout-note appearance="simple"}
**Posição no estudo:** esta etapa produz `datasets/whole_century.csv`, usado pela [curadoria e limpeza](cleaning.ipynb).
:::

## Dependências

In [2]:
import re
import pandas as pd
from bs4 import BeautifulSoup
import requests

## Leitura da fonte

A coleta aponta para a página registrada no projeto. Como seletores e posições dependem da estrutura do HTML externo, uma mudança na página pode exigir revisão antes de uma nova execução.

In [3]:
url = ("https://tvglobo.fandom.com/pt-br/wiki/Lista_de_filmes_exibidos_na_Sess%C3%A3o_da_Tarde#2022")

html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser") # parser - processa

In [4]:
output = soup("div", "mw-parser-output")
output = output[0]

A célula seguinte funciona como uma conferência pontual da associação entre a lista, o mês e o ano encontrados no HTML.

In [5]:
anos = output.find_all('h2')
meses = output.find_all('h3')
filmes = output.find_all('ul')[51:]
anos = anos[1:]
#anos
#meses
#filmes
#meses[0].findNextSibling('ul')

## Estruturação dos registros

In [6]:
filmes[6].findPreviousSibling('h3').findPreviousSibling('h2').text

'2023'

In [7]:
lista_filmes = []

Para cada item, o notebook separa a data, o título e eventuais comentários entre parênteses. O resultado é convertido em um `DataFrame`.

In [8]:
for lista in filmes:
    for filme in lista.find_all('li'):
        ano = lista.findPreviousSibling('h2').text
        # print(lista.findPreviousSibling('h3').text) # - mes
        # print(filme.text)

        data = filme.text[:5]
        data = data.replace('/', '-')

        data = f'{ano}-{data[-2:]}-{data[:-3]}' # 2000-12-31
        titulo = filme.text[8:]

        if '(' in titulo:
            padrao = r"\((.*?)\)"

            comment = re.search(padrao, titulo).group(1)
            titulo = re.sub(padrao, "", titulo)
        else:
            comment = ''

        filme_info = {
            'date': data,
            'title': titulo,
            'comments': comment
        }

        lista_filmes.append(filme_info)

In [10]:
df = pd.DataFrame(lista_filmes)
df.to_csv('datasets/whole_century.csv', index=False)

::: {.callout-warning appearance="simple"}
A versão publicada usa os resultados e CSVs já versionados; ela não refaz a requisição durante o build do site. Isso evita que uma alteração temporária na fonte interrompa a publicação.
:::

[Continuar para a curadoria e limpeza →](cleaning.ipynb){.btn .btn-primary}